Python Tutorial - GeeksforGeeks: The Ultimate Spoon-Feeding of Python

In [1]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI
from bs4 import BeautifulSoup
import requests

In [2]:
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


In [3]:
openai = OpenAI()

In [10]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = """
You are a senior technical content analyst with a sarcastic, witty tone.

You analyze web pages from learning platforms like GeeksforGeeks, especially programming tutorials.

Your job:
- Ignore navigation menus, ads, footer, and unrelated boilerplate text.
- Focus ONLY on actual educational content (tutorials, explanations, examples, headings).
- Identify the structure of the article (sections, topics, subtopics).
- Provide a detailed but engaging summary.

Style:
- Slightly snarky and humorous, but not excessive.
- Still informative and complete.
- Use markdown formatting (headings + bullet points where useful).
- Do NOT wrap output in code blocks.
"""

In [11]:
# Define our user prompt

user_prompt_prefix = """
You are given raw text extracted from a GeeksforGeeks page.

Task:
1. Identify the topic of the page.
2. Extract key sections covered in the tutorial.
3. Summarize each section briefly but clearly.
4. Mention practical concepts (code, DS, algorithms, etc.).
5. End with a short witty one-liner summary.

Important:
- Ignore navigation, ads, unrelated links, and repeated headers.
- Do not copy text verbatim.
- Be structured and moderately detailed (not one paragraph).

"""

In [6]:
# Scraper to fetch Python tutorial from GeeksforGeeks website

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/117.0.0.0 Safari/537.36"
    )
}


def fetch_website_contents(url, max_chars=3000):
    """
    Fetch and clean webpage text content.
    """

    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        response.raise_for_status()

        soup = BeautifulSoup(response.content, "html.parser")

        # Extract title
        title = soup.title.string.strip() if soup.title else "No title"

        # Remove unwanted tags
        for tag in soup([
            "script",
            "style",
            "img",
            "svg",
            "noscript",
            "footer",
            "header",
            "nav",
            "aside",
            "form",
            "button",
        ]):
            tag.decompose()

        # Extract main text
        text = soup.get_text(separator="\n", strip=True)

        # Remove empty lines
        lines = [line.strip() for line in text.splitlines()]
        cleaned_text = "\n".join(line for line in lines if line)

        final_content = f"TITLE:\n{title}\n\nCONTENT:\n{cleaned_text}"

        return final_content[:max_chars]

    except requests.exceptions.RequestException as e:
        return f"Error fetching website: {e}"

In [12]:


def build_messages(website_content):
    return [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": f"{user_prompt_prefix}\n\n{website_content}"
        }
    ]

def summarize(url, model="gpt-4.1-mini"):
    # Fetch website content
    website_content = fetch_website_contents(url)

    # Create messages payload
    messages = build_messages(website_content)

    # Call OpenAI API
    response = openai.chat.completions.create(
        model=model,
        messages=messages
    )

    # Return summary text
    return response.choices[0].message.content


def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))


# Example usage
display_summary("https://www.geeksforgeeks.org/python-programming-language-tutorial/")

# Analysis of "Python Tutorial - GeeksforGeeks"

## 1. Topic of the Page
The page is a comprehensive **Python programming tutorial** from GeeksforGeeks, designed for beginners and intermediate learners who want to understand Python's syntax, features, and practical applications ranging from basics to advanced topics like OOP and data structures.

---

## 2. Key Sections Covered

- Introduction to Python: Overview, reasons to learn, usage
- Basics: Installation, writing first program, variables, operators, keywords, data types, control flow (conditions, loops)
- Functions: Syntax, scope, advanced function concepts (lambda, decorators, recursion)
- Data Structures: Python’s built-in types and collections
- Object-Oriented Programming (OOP): Concepts, classes, polymorphism

---

## 3. Section Summaries

### Introduction to Python
- Introduction to Python's popularity owing to its simplicity, readability, and extensive libraries.
- Explains Python's versatility across fields like AI, web development, and automation.
- Basic syntax example given (classic "Hello, World!") showing the `print()` function.
- Reasons to learn Python, highlighting its fewer lines of code, multiple frameworks, and broad industry use (Google, Netflix, NASA).

### Basics
- Walks through installing Python and writing the first program.
- Introduces fundamental programming concepts: variables, keywords, operators.
- Covers primary data types and explains conditional statements and loops.
- Empowers learners with essentials for writing simple scripts.

### Functions
- Delves into Python function declarations, parameters, return values, and variable scope.
- Explores common and advanced functional programming constructs:
    - `*args` and `**kwargs` for flexible arguments
    - First-class functions (functions treated like variables)
    - Lambda (anonymous) functions for quick, concise operations
    - Functional tools like `map()`, `filter()`, `reduce()`
    - Decorators for modifying function behavior
    - Recursion and inner functions

### Data Structures
- Detailed treatment of Python’s built-in data types:
    - Strings, Lists, Tuples, Dictionaries, Sets, Arrays
- Introduction to list comprehensions for elegant, compact loops.
- Overview of specialized data structures in the `collections` module:
    - Counters for counting hashable objects
    - Heapq for priority queues
    - Deque for fast appends/pops at both ends
    - OrderedDict and DefaultDict for enhanced dictionary behaviors

### Object-Oriented Programming (OOP)
- Focus on core OOP concepts to build modular, reusable code:
    - Classes and objects as building blocks
    - The elusive `self` parameter and its default nature in methods
    - Polymorphism for dynamic method behavior
    - Abstract classes and iterators (though mentioned partially)
- Equips learners to architect scalable Python applications.

---

## 4. Practical Concepts Highlighted
- **Code examples:** Starts simple (`print("Hello World!")`) progressing to complex topics.
- **Fundamental programming constructs:** Variables, operators, data types, flow control.
- **Functions:** Defining, passing arguments, lambda expressions, recursion, decorators.
- **Data structures:** Mastering Python’s native types plus `collections` module usage.
- **Algorithmic tools:** Functional programming aids like `map`, `filter`, and `reduce`.
- **OOP principles:** Classes, objects, polymorphism, encapsulation.
- **Real-world application readiness** through frameworks and libraries mentions (Django, Flask, Pandas, TensorFlow).

---

## 5. Witty One-Liner Summary
Learning Python the GeeksforGeeks way: because writing less code to do more—you know, the lazy genius’s dream—is basically just showing off with style.